In [7]:
import pandas as pd
import altair as alt

In [8]:
data = pd.read_csv("../../data/raw/stocks.csv")
data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Close_BROS,Close_COFF.L,...,Open_SBUX,Open_SJM,Stock Splits_BROS,Stock Splits_COFF.L,Stock Splits_SBUX,Stock Splits_SJM,Volume_BROS,Volume_COFF.L,Volume_SBUX,Volume_SJM
0,2025-03-18,386.000000,391.950012,383.700012,387.149994,22,0.0,0.0,63.009998,67.415001,...,96.565563,107.437419,0.0,0.0,0.0,0.0,2499000.0,2167.0,7423400.0,1249600.0
1,2025-03-19,394.149994,394.149994,394.100006,394.600006,12747,0.0,0.0,66.290001,68.370003,...,95.631332,105.687981,0.0,0.0,0.0,0.0,5356200.0,4887.0,4982700.0,1292600.0
2,2025-03-20,389.299988,397.600006,386.049988,392.149994,14036,0.0,0.0,65.489998,68.864998,...,96.195758,105.966729,0.0,0.0,0.0,0.0,4395400.0,458.0,5952800.0,1185300.0
3,2025-03-21,392.000000,395.750000,386.500000,391.399994,12375,0.0,0.0,65.430000,68.349998,...,95.310189,104.947822,0.0,0.0,0.0,0.0,2807100.0,1021.0,11366000.0,2326900.0
4,2025-03-24,391.450012,402.399994,390.600006,393.399994,13306,0.0,0.0,70.449997,68.894997,...,95.397767,105.755255,0.0,0.0,0.0,0.0,4607500.0,2286.0,8265000.0,1427200.0


In [9]:
# Identify all columns that are close prices
close_cols = [col for col in data.columns if col.startswith('Close')]
if 'Close' not in close_cols and 'Close' in data.columns:
    close_cols.insert(0, 'Close')

# Melt the DataFrame for Altair's multi-line plotting
plot_data = data.melt(id_vars=['Date'], value_vars=close_cols, 
                      var_name='Symbol', value_name='ClosePrice')

# Plot all close price lines over time
chart = alt.Chart(plot_data).mark_line().encode(
    x='Date:T',
    y='ClosePrice:Q',
    color='Symbol:N'
).properties(
    title='Close Prices Over Time',
    width=1200,
    height=400
).configure_view(
    strokeWidth=0
)
chart

alt.Chart(...)

In [10]:
# Four scatterplots comparing each of the other close prices to the main
import itertools

main_close_col = 'Close'

# Find all columns with close prices except the main one
other_close_cols = [col for col in close_cols if col != main_close_col]

scatter_charts = []
for col in other_close_cols:
    scatter_df = data[[main_close_col, col]].dropna()

    # Compute the min/max for each axis and add some padding/margin
    x_min, x_max = scatter_df[main_close_col].min(), scatter_df[main_close_col].max()
    y_min, y_max = scatter_df[col].min(), scatter_df[col].max()

    x_range = x_max - x_min
    y_range = y_max - y_min

    x_pad = x_range * 0.05 if x_range else 1
    y_pad = y_range * 0.05 if y_range else 1

    x_scale = alt.Scale(domain=[x_min - x_pad, x_max + x_pad], zero=False)
    y_scale = alt.Scale(domain=[y_min - y_pad, y_max + y_pad], zero=False)

    scatter = (
        alt.Chart(scatter_df)
        .mark_circle(size=60, opacity=0.7)
        .encode(
            x=alt.X(main_close_col, title='Main Close Price', scale=x_scale),
            y=alt.Y(col, title=f'{col}', scale=y_scale),
            tooltip=[main_close_col, col]
        )
        .properties(
            width=300,
            height=300,
            title=f'{col} vs Main Close'
        )
    )
    scatter_charts.append(scatter)

# Display all four scatterplots side by side
(scatter_charts[0] | scatter_charts[1]) & (scatter_charts[2] | scatter_charts[3])

alt.VConcatChart(...)

In [11]:
import numpy as np

# Compute the correlation matrix for all the close columns
correlation_matrix = data[close_cols].corr()

# Display the correlation matrix as a DataFrame
print("Correlation matrix of close prices:")
display(correlation_matrix)

# Optional: visualize as a heatmap using Altair
import altair as alt
import pandas as pd

corr_df = correlation_matrix.reset_index().melt(id_vars='index')
corr_df.columns = ['Close_1', 'Close_2', 'Correlation']

heatmap = (
    alt.Chart(corr_df)
    .mark_rect()
    .encode(
        x=alt.X('Close_1:O', title='Close Price 1'),
        y=alt.Y('Close_2:O', title='Close Price 2'),
        color=alt.Color('Correlation:Q', scale=alt.Scale(scheme='redblue', domain=(-1, 1))),
        tooltip=['Close_1', 'Close_2', alt.Tooltip('Correlation:Q', format='.2f')]
    )
    .properties(width=300, height=300, title="Correlation Matrix Heatmap of Close Prices")
)

heatmap.display()

Correlation matrix of close prices:


,Close,Close_BROS,Close_COFF.L,Close_SBUX,Close_SJM
Close,1.000000,0.064203,0.918696,-0.676531,-0.020831
Close_BROS,0.064203,1.000000,-0.133781,-0.027553,-0.163074
Close_COFF.L,0.918696,-0.133781,1.000000,-0.547313,-0.122110
Close_SBUX,-0.676531,-0.027553,-0.547313,1.000000,0.050653
Close_SJM,-0.020831,-0.163074,-0.122110,0.050653,1.000000


alt.Chart(...)